In [ ]:
%reload_ext autoreload
%autoreload 2

import os
from pathlib import Path

print(Path().cwd())
os.chdir(Path(os.getcwd()).parent)
print(Path().cwd())

## Select Contrast-Enhanced Ultrasound (CEUS) Cine and Parser

In [ ]:
from src.image_loading.options import get_scan_loaders

print("Available scan loaders:", list(get_scan_loaders().keys()))

In [ ]:
scan_type = 'nifti'

scan_path = '/home/yuanshanwu/Documents/TUL/CEUS-Studies/P05_V02_CE01/NewInterpolation/UCSD-P05-V02-CE1_09.45.08/UCSD-P05-V02-CE1_09.45.08_mf_sip_capture_50_2_1_0_CEUS.nii'
bmode_scan_path = '/home/yuanshanwu/Documents/TUL/CEUS-Studies/P05_V02_CE01/NewInterpolation/UCSD-P05-V02-CE1_09.45.08/UCSD-P05-V02-CE1_09.45.08_mf_sip_capture_50_2_1_0_BMODE.nii'
scan_loader_kwargs = {
    'transpose': False,
}

In [ ]:
from src.entrypoints import scan_loading_step

image_data = scan_loading_step(scan_type, scan_path, **scan_loader_kwargs)
bmode_image_data = scan_loading_step(scan_type, bmode_scan_path, **scan_loader_kwargs)

In [ ]:
image_data.pixel_data.shape

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

axial_slice = bmode_image_data.pixel_data[:, :, 105, 0]  # (X, Y) of first Z and T
# Mask out zeros before computing percentiles
non_zero = axial_slice[axial_slice != 0]

# Compute 10th and 90th percentile on non-zero values
p_low = np.percentile(non_zero, 2)
p_high = np.percentile(non_zero, 98)

# Clip and normalize to 0-255
clipped_slice = np.clip(axial_slice, p_low, p_high)
normalized_slice = ((clipped_slice - p_low) / (p_high - p_low) * 255).astype(np.uint8)

# Display all three side by side
fig, axes = plt.subplots(1, 2, figsize=(20, 6))

axes[0].imshow(axial_slice.T, cmap='gray')
axes[0].set_title('Original')
axes[0].axis('off')
plt.colorbar(axes[0].images[0], ax=axes[0], label='Pixel Intensity')

axes[1].imshow(normalized_slice.T, cmap='gray')
axes[1].set_title(f'Clipped [{p_low:.1f}, {p_high:.1f}] → Normalized [0, 255]')
axes[1].axis('off')
plt.colorbar(axes[1].images[0], ax=axes[1], label='Pixel Intensity')

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter, sobel
import cv2
from ipywidgets import interactive, IntSlider, FloatSlider, VBox, HBox, Output, HTML, Label
import ipywidgets as widgets
from IPython.display import display

# ============================================================
# FIXED ADAPTIVE SPECKLE REDUCTION
# ============================================================

def adaptive_speckle_reduction_auto(image_uint8, edge_percentile=85, 
                                    smooth_strength=7, edge_sigma=1.0):
    """
    FIXED: Percentile-based edge detection
    """
    
    img = image_uint8.astype(np.float32)
    
    # Calculate gradient
    gradient_x = sobel(img, axis=1)
    gradient_y = sobel(img, axis=0)
    gradient_mag = np.sqrt(gradient_x**2 + gradient_y**2)
    gradient_smooth = gaussian_filter(gradient_mag, sigma=edge_sigma)
    
    # Auto-threshold using percentile
    mask = image_uint8 > 0
    if mask.any():
        edge_threshold = np.percentile(gradient_smooth[mask], edge_percentile)
    else:
        edge_threshold = np.percentile(gradient_smooth, edge_percentile)
    
    # Create edge mask: High gradient = EDGE (preserve)
    edge_mask = (gradient_smooth > edge_threshold).astype(np.float32)
    kernel = np.ones((3, 3), np.uint8)
    edge_mask = cv2.dilate(edge_mask, kernel, iterations=1)
    
    # Smooth speckle regions
    smoothed = cv2.bilateralFilter(
        img.astype(np.uint8),
        d=int(smooth_strength),
        sigmaColor=smooth_strength * 10,
        sigmaSpace=smooth_strength * 10
    )
    
    # Blend: edges keep original, speckle gets smoothed
    edge_mask_smooth = gaussian_filter(edge_mask, sigma=2.0)
    result = edge_mask_smooth * img + (1 - edge_mask_smooth) * smoothed
    
    return np.clip(result, 0, 255).astype(np.uint8), edge_mask, gradient_smooth, edge_threshold


# ============================================================
# INTERACTIVE GUI WITH LARGE VISIBLE SLIDERS
# ============================================================

def speckle_tuner(image_uint8):
    """
    Interactive speckle tuner with LARGE VISIBLE SLIDERS
    
    Usage:
        speckle_tuner(normalized_slice)
    """
    
    # Output widget for plots
    out = Output()
    
    def update_plot(edge_percentile, smooth_strength, edge_sigma):
        with out:
            out.clear_output(wait=True)
            
            # Process image
            result, edge_mask, gradient, threshold = adaptive_speckle_reduction_auto(
                image_uint8, edge_percentile, smooth_strength, edge_sigma
            )
            
            # Calculate statistics
            mask = image_uint8 > 0
            edge_pct = edge_mask[mask].sum() / mask.sum() * 100 if mask.any() else 0
            speckle_pct = 100 - edge_pct
            
            # Create figure
            fig = plt.figure(figsize=(20, 12))
            
            # Row 1: Full images
            ax1 = plt.subplot(2, 4, 1)
            ax1.imshow(image_uint8.T, cmap='gray', vmin=0, vmax=255)
            ax1.set_title('Original B-mode', fontsize=12, fontweight='bold')
            ax1.axis('off')
            
            ax2 = plt.subplot(2, 4, 2)
            im_grad = ax2.imshow(gradient.T, cmap='hot', vmin=0, vmax=np.percentile(gradient[mask], 99) if mask.any() else None)
            ax2.set_title(f'Gradient Magnitude\n(Threshold={threshold:.1f})', fontsize=11)
            ax2.axis('off')
            plt.colorbar(im_grad, ax=ax2, fraction=0.046)
            
            ax3 = plt.subplot(2, 4, 3)
            ax3.imshow(edge_mask.T, cmap='RdYlGn_r')
            ax3.set_title(f'Edge Mask\n(Red=Edge {edge_pct:.1f}%, Green=Speckle {speckle_pct:.1f}%)', fontsize=11)
            ax3.axis('off')
            
            ax4 = plt.subplot(2, 4, 4)
            ax4.imshow(result.T, cmap='gray', vmin=0, vmax=255)
            ax4.set_title('✅ Despeckled Result', fontsize=12, fontweight='bold', color='green')
            ax4.axis('off')
            
            # Row 2: Zoomed views
            h, w = image_uint8.shape
            zoom_y, zoom_x = slice(h//3, 2*h//3), slice(w//3, 2*w//3)
            
            ax5 = plt.subplot(2, 4, 5)
            ax5.imshow(image_uint8[zoom_x, zoom_y].T, cmap='gray', vmin=0, vmax=255)
            ax5.set_title('Original (Zoomed)', fontsize=10)
            ax5.axis('off')
            
            ax6 = plt.subplot(2, 4, 6)
            ax6.imshow(gradient[zoom_x, zoom_y].T, cmap='hot')
            ax6.set_title('Gradient (Zoomed)', fontsize=10)
            ax6.axis('off')
            
            ax7 = plt.subplot(2, 4, 7)
            ax7.imshow(edge_mask[zoom_x, zoom_y].T, cmap='RdYlGn_r')
            ax7.set_title('Mask (Zoomed)', fontsize=10)
            ax7.axis('off')
            
            ax8 = plt.subplot(2, 4, 8)
            ax8.imshow(result[zoom_x, zoom_y].T, cmap='gray', vmin=0, vmax=255)
            ax8.set_title('Result (Zoomed)', fontsize=10)
            ax8.axis('off')
            
            plt.suptitle(f'📊 Current: Edge={edge_percentile}% | Smooth={smooth_strength} | Sigma={edge_sigma:.1f}',
                        fontsize=14, fontweight='bold')
            plt.tight_layout()
            plt.show()
    
    # ========================================================
    # CREATE LARGE VISIBLE SLIDERS
    # ========================================================
    
    # Slider 1: Edge Percentile
    edge_label = Label(value='🎚️ Edge Percentile (70=preserve edges | 95=smooth more):')
    edge_slider = IntSlider(
        value=85,
        min=70,
        max=95,
        step=1,
        description='',
        continuous_update=False,  # Only update when you release the slider
        orientation='horizontal',
        readout=True,
        readout_format='d',
        layout=widgets.Layout(width='800px', height='50px'),
        style={'handle_color': 'steelblue', 'description_width': '0px'}
    )
    edge_value = Label(value=f'Current value: 85')
    
    # Slider 2: Smooth Strength
    smooth_label = Label(value='🎚️ Smooth Strength (3=gentle | 15=strong):')
    smooth_slider = IntSlider(
        value=7,
        min=3,
        max=15,
        step=1,
        description='',
        continuous_update=False,
        orientation='horizontal',
        readout=True,
        layout=widgets.Layout(width='800px', height='50px'),
        style={'handle_color': 'forestgreen'}
    )
    smooth_value = Label(value=f'Current value: 7')
    
    # Slider 3: Edge Sigma
    sigma_label = Label(value='🎚️ Edge Smoothness (0.5=sharp | 3.0=soft):')
    sigma_slider = FloatSlider(
        value=1.0,
        min=0.5,
        max=3.0,
        step=0.1,
        description='',
        continuous_update=False,
        orientation='horizontal',
        readout=True,
        readout_format='.1f',
        layout=widgets.Layout(width='800px', height='50px'),
        style={'handle_color': 'coral'}
    )
    sigma_value = Label(value=f'Current value: 1.0')
    
    # Update value labels
    def update_edge_label(change):
        edge_value.value = f'Current value: {change.new}'
        update_plot(edge_slider.value, smooth_slider.value, sigma_slider.value)
    
    def update_smooth_label(change):
        smooth_value.value = f'Current value: {change.new}'
        update_plot(edge_slider.value, smooth_slider.value, sigma_slider.value)
    
    def update_sigma_label(change):
        sigma_value.value = f'Current value: {change.new:.1f}'
        update_plot(edge_slider.value, smooth_slider.value, sigma_slider.value)
    
    edge_slider.observe(update_edge_label, 'value')
    smooth_slider.observe(update_smooth_label, 'value')
    sigma_slider.observe(update_sigma_label, 'value')
    
    # ========================================================
    # CREATE UI LAYOUT
    # ========================================================
    
    title = HTML(
        value="""
        <div style='background-color: #4682b4; padding: 20px; border-radius: 10px; margin-bottom: 20px;'>
            <h1 style='color: white; text-align: center; margin: 0;'>🎛️ SPECKLE REDUCTION TUNER</h1>
            <p style='color: white; text-align: center; margin: 10px 0 0 0; font-size: 16px;'>
                Adjust the sliders below and see results update in real-time
            </p>
        </div>
        """
    )
    
    instructions = HTML(
        value="""
        <div style='background-color: #fff3cd; padding: 15px; border-radius: 8px; margin-bottom: 20px; border: 2px solid #ffc107;'>
            <h3 style='margin-top: 0; color: #856404;'>📖 Quick Guide:</h3>
            <ul style='color: #856404; font-size: 14px;'>
                <li><b>Edge Percentile:</b> Higher value = smoother image (more speckle removed)</li>
                <li><b>Smooth Strength:</b> Higher value = stronger smoothing in speckle regions</li>
                <li><b>Edge Smoothness:</b> Higher value = softer edge detection</li>
            </ul>
            <p style='color: #856404; margin-bottom: 0;'><b>💡 Recommended start:</b> Edge=85, Smooth=7, Sigma=1.0</p>
        </div>
        """
    )
    
    footer = HTML(
        value="""
        <div style='background-color: #d1ecf1; padding: 15px; border-radius: 8px; margin-top: 20px; border: 2px solid #0c5460;'>
            <p style='text-align: center; color: #0c5460; margin: 0; font-size: 14px;'>
                <b>When done tuning, run:</b> <code style='background-color: #fff; padding: 3px 8px; border-radius: 3px;'>best_params = get_speckle_params()</code>
            </p>
        </div>
        """
    )
    
    # Assemble UI
    ui = VBox([
        title,
        instructions,
        HTML('<div style="height: 30px;"></div>'),  # Spacer
        edge_label,
        edge_slider,
        edge_value,
        HTML('<div style="height: 20px;"></div>'),  # Spacer
        smooth_label,
        smooth_slider,
        smooth_value,
        HTML('<div style="height: 20px;"></div>'),  # Spacer
        sigma_label,
        sigma_slider,
        sigma_value,
        HTML('<div style="height: 30px;"></div>'),  # Spacer
        footer,
        out
    ])
    
    # Display the UI
    display(ui)
    
    # Initial plot
    update_plot(85, 7, 1.0)
    
    # Function to get current parameters
    def get_params():
        params = {
            'edge_percentile': edge_slider.value,
            'smooth_strength': smooth_slider.value,
            'edge_sigma': sigma_slider.value
        }
        print("\n" + "="*70)
        print("✅ CURRENT PARAMETERS:")
        print("="*70)
        print(f"  edge_percentile:  {params['edge_percentile']}")
        print(f"  smooth_strength:  {params['smooth_strength']}")
        print(f"  edge_sigma:       {params['edge_sigma']:.1f}")
        print("="*70)
        print("\n📝 To apply these parameters:")
        print(f"\nresult, edge_map, gradient, threshold = adaptive_speckle_reduction_auto(")
        print(f"    normalized_slice,")
        print(f"    edge_percentile={params['edge_percentile']},")
        print(f"    smooth_strength={params['smooth_strength']},")
        print(f"    edge_sigma={params['edge_sigma']:.1f}")
        print(f")")
        print("="*70)
        return params
    
    # Make globally accessible
    global get_speckle_params
    get_speckle_params = get_params
    
    print("\n✅ GUI loaded! Adjust the sliders above to tune parameters.")
    print("💡 When done, run: best_params = get_speckle_params()")


# ============================================================
# USAGE - ONE LINE!
# ============================================================

speckle_tuner(normalized_slice)

In [ ]:
%matplotlib qt
best_params = speckle_tuner(normalized_slice)

In [ ]:
## CEUS

import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import median_filter

ref_slice = image_data.pixel_data[:, :, 100, 0]
axial_slice = image_data.pixel_data[:, :, 100, 50]

# Mask out zeros before computing percentiles
non_zero = axial_slice[axial_slice != 0]
non_zero_ref = ref_slice[ref_slice != 0]

p_low = np.mean(non_zero_ref)
p_high = np.percentile(non_zero, 100)

# Clip and normalize to 0-255
clipped_slice = np.clip(axial_slice, p_low, p_high)
normalized_slice = ((clipped_slice - p_low) / (p_high - p_low) * 255).astype(np.uint8)

# Display all three side by side
fig, axes = plt.subplots(1, 2, figsize=(20, 6))

axes[0].imshow(axial_slice.T, cmap='gray')
axes[0].set_title('Original')
axes[0].axis('off')
plt.colorbar(axes[0].images[0], ax=axes[0], label='Pixel Intensity')

axes[1].imshow(normalized_slice.T, cmap='gray')
axes[1].set_title(f'Clipped [{p_low:.1f}, {p_high:.1f}] → Normalized [0, 255]')
axes[1].axis('off')
plt.colorbar(axes[1].images[0], ax=axes[1], label='Pixel Intensity')

plt.tight_layout()
plt.show()

## Image Preprocessing

In [ ]:
from src.image_preprocessing.options import get_im_preproc_funcs, get_required_im_preproc_kwargs

print("Available preprocessing functions:", list(get_im_preproc_funcs().keys()))

In [ ]:
preproc_func_names = ['resample'] # in order of application
required_kwargs = get_required_im_preproc_kwargs(preproc_func_names)
print("Required kwargs for preprocessing functions:", required_kwargs)

In [ ]:
preproc_kwargs = {
    'target_vox_size': (1.0, 1.0, 1.0),
    'interp': 'linear',
}

In [ ]:
from src.entrypoints import scan_preprocessing_step

image_data = scan_preprocessing_step(preproc_func_names, image_data, **preproc_kwargs)

## Load Segmentation

Assumes same segmentation for each frame

In [ ]:
from src.seg_loading.options import get_seg_loaders

print("Available segmentation loaders:", list(get_seg_loaders().keys()))

In [ ]:
seg_type = 'nifti'

seg_path = '/media/das/TOSHIBA EXT/P-Selectin Data/July VOIs (Ashley)/20190725103303.756_segmentation_LEFT.nii.gz'
seg_loader_kwargs = {}

In [ ]:
from src.entrypoints import seg_loading_step

seg_data = seg_loading_step(seg_type, image_data, seg_path, scan_path, **seg_loader_kwargs)

## Segmentation Preprocessing

In [ ]:
from src.seg_preprocessing.options import get_seg_preproc_funcs, get_required_seg_preproc_kwargs

print("Available preprocessing functions:", list(get_seg_preproc_funcs().keys()))

In [ ]:
preproc_func_names = ['resample'] # in order of application
required_kwargs = get_required_seg_preproc_kwargs(preproc_func_names)
print("Required kwargs for preprocessing functions:", required_kwargs)

In [ ]:
preproc_kwargs = {
    'target_vox_size': (1.0, 1.0, 1.0),
    'interp': 'nearest',
}

In [ ]:
from src.entrypoints import seg_preprocessing_step

seg_data = seg_preprocessing_step(preproc_func_names, image_data, seg_data, **preproc_kwargs)

## CEUS Quantitative Temporal Curve Analysis

In [ ]:
from src.time_series_analysis.options import get_analysis_types, get_required_kwargs

all_analysis_types, all_analysis_funcs = get_analysis_types()
print("Available analysis types:", list(all_analysis_types.keys()))

In [ ]:
analysis_type = 'curves'

print("Available analysis functions:", list(all_analysis_funcs.keys()))

In [ ]:
analysis_funcs = ['tic']

# Find all required kwargs for the analysis functions
analysis_funcs = analysis_funcs if len(analysis_funcs) else list(all_analysis_funcs[analysis_type].keys())
required_kwargs = get_required_kwargs(analysis_type, analysis_funcs)
print("Required kwargs for current analysis:", required_kwargs)

In [ ]:
analysis_kwargs = {
    'curves_output_path': 'sample.csv',
}

In [ ]:
from src.entrypoints import analysis_step

analysis_obj = analysis_step(analysis_type, image_data, seg_data, analysis_funcs, **analysis_kwargs)

## Curve Quantification

In [ ]:
from src.curve_quantification.options import get_quantification_funcs

quantification_funcs = get_quantification_funcs()
print("Available quantification functions:", quantification_funcs.keys())

In [ ]:
function_names = ['dte'] # Empty list will use all functions
output_path = 'test_quants.csv'
curve_quantifications_kwargs = {
    'curves_to_fit': ['moderate-pselectin_diagnostics_Image-original_Mean'],
    'n_frames_to_analyze': 100,
    'tic_name': 'TIC'
}

In [ ]:
from src.entrypoints import curve_quantification_step

curve_quant = curve_quantification_step(analysis_obj, function_names, output_path, **curve_quantifications_kwargs)